# Ablations on top of the final model, and the complete ablation table

| ID | Change vs final model | Paper | Datasets |
|---|---|---|---|
| ST-D | **depth only**: segmentation loss weight 0 (`w_seg=0`, which also removes the aux head's loss) | SwinMTL Tab. IV (single- vs multi-task) | NYUv2, Cityscapes |
| ST-S | **segmentation only**: depth loss weight 0 (`w_depth=0`) | SwinMTL Tab. IV | NYUv2, Cityscapes |
| PM | **patch mixing off** (`use_patch_mix=False`) | SwinMTL §IV-A | NYUv2 |
| FPE-NYU | **frame-consistent position embeddings off** on NYU (AB1 was only tested on Cityscapes) | – (our AB1) | NYUv2 |
| D4 | **DINOv2's depth head**: 256 uniformly spaced bins in [d_min, d_max] with linear normalisation (relu + 0.1, divided by the sum, following AdaBins), depth = Σ p_i c_i, trained with the same SiLog loss. It replaces our sigmoid output (linear or log). | DINOv2 §7.4 (`dinov2/hub/depth/decode_heads.py`, `BNHead`: "UD" bins, "linear" normalisation) | NYUv2 |

**D4 is a side ablation only**. It is never part of the final model, whatever its result.

**Baselines.** Cityscapes rows: the final Cityscapes model (seed 42). NYU rows: ST-D, ST-S and PM are measured against the NYU final model (notebook 06 8-block arm, section 'NYU ablations re-based on the NYU final model'); FPE-NYU and D4 against notebook 09's NYU configuration (frame PE on). PM on 3 seeds is notebook 12.

## Setup

***NOTE:*** The DINOv2 and DPT code is used straight from the local repositories `./dinov2` and `./DPT`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [2]:
from mtl_final import *   # training code + decision rule, sigma, train_eval, load_eval (as in notebook 09)

configure(results_root=ROOT / "results")

CONFIG = load_json(FINAL_DIR / "config_final.json")
FINAL_TAG = "log" if CONFIG["depth_param"] == "log" else "base"
print("final configuration:", CONFIG, "| tag:", FINAL_TAG)
R_FINAL42 = {d: train_eval(d, 42, FINAL_TAG, CONFIG) for d in ("nyu", "cityscapes")}   # cached: only loads



sigma: {'rmse': 0.003, 'miou': 0.2, 'abs_rel_cv': 0.01, 'source': 'notebook 06 (raw: rmse=0.001669, miou=0.1389, abs_rel_cv=0.005997)'}
final configuration: {'blocks': 8, 'use_critic': False, 'frame_pe': True, 'photo_rng_fix': True, 'depth_param': 'linear', 'nyu_scale_aug': [1.0, 1.5], 'nyu_crop': None, 'nyu_rotate_deg': 0.0} | tag: base
=== nyu base seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base_s42
nyu_final: already trained
=== cityscapes base seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/cityscapes_base_s42
cityscapes_final: already trained


## Runs

In [ ]:
ABLATIONS = [
    # (id, dataset, tag suffix, configure overrides)
    ("FPE-NYU", "nyu", "noframepe", dict(frame_pe=False)),
    ("ST-D", "cityscapes", "depthonly", dict(w_seg=0.0)),
    ("ST-S", "cityscapes", "segonly", dict(w_depth=0.0)),
    ("ST-D", "nyu", "depthonly", dict(w_seg=0.0)),
    ("ST-S", "nyu", "segonly", dict(w_depth=0.0)),
    ("PM", "nyu", "nopatchmix", dict(use_patch_mix=False)),
    ("D4", "nyu", "bins", dict(depth_head="bins", depth_param="linear")),
]
R_ABL = {}
for abl_id, dataset, suffix, extra in ABLATIONS:
    R_ABL[(abl_id, dataset)] = train_eval(dataset, 42, f"{FINAL_TAG}-{suffix}", CONFIG, **extra)


=== nyu base-noframepe seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-noframepe_s42
nyu_final: already trained
=== cityscapes base-depthonly seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/cityscapes_base-depthonly_s42
cityscapes_final: already trained
=== cityscapes base-segonly seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/cityscapes_base-segonly_s42
cityscapes_final: already trained
=== nyu base-depthonly seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-depthonly_s42
nyu_final: already trained
=== nyu base-segonly seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-segonly_s42
nyu_final: already trained
=== nyu base-nopatchmix seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nopatchmix_s42
nyu_final: already trained
=== nyu base-bins seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-bins_s42
nyu_final: already trained


### NYU ablations re-based on the NYU final model

In [ ]:
NYU_A = dict(frame_pe=False, photo_rng_fix=False)
R_NYU_A42 = load_eval(ROOT / "results" / "seeds" / "nc8_s42", "nyu_partial_ft8")
print(f"NYU final model (option A), seed 42: RMSE {R_NYU_A42['nyu_depth']['rmse']:.4f}, mIoU {R_NYU_A42['nyu_seg']['mIoU']:.2f}")
ABLATIONS_A = [
    ("ST-D", "nyu", "nofpe-depthonly", dict(NYU_A, w_seg=0.0)),
    ("ST-S", "nyu", "nofpe-segonly", dict(NYU_A, w_depth=0.0)),
    ("PM", "nyu", "nofpe-nopatchmix", dict(NYU_A, use_patch_mix=False)),
]
R_ABL_A = {}
for abl_id, dataset, suffix, extra in ABLATIONS_A:
    R_ABL_A[(abl_id, dataset)] = train_eval(dataset, 42, f"{FINAL_TAG}-{suffix}", CONFIG, **extra)


NYU final model (option A), seed 42: RMSE 0.4205, mIoU 58.25
=== nyu base-nofpe-depthonly seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nofpe-depthonly_s42
nyu_final: already trained
=== nyu base-nofpe-segonly seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nofpe-segonly_s42
nyu_final: already trained
=== nyu base-nofpe-nopatchmix seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nofpe-nopatchmix_s42
nyu_final: already trained


## Results vs the final model (seed 42)
For single-task runs only the trained task is compared; the other column is left empty.

In [ ]:
SUPERSEDED = {("ST-D", "nyu"), ("ST-S", "nyu"), ("PM", "nyu")}
runs = [(a, d, R_FINAL42[d], R_ABL[(a, d)],
         "notebook 09 NYU config (frame PE on)" + (", superseded by §1b" if (a, d) in SUPERSEDED else "") if d == "nyu" else "final model")
        for a, d, _, _ in ABLATIONS]
runs += [(a, d, R_NYU_A42, R_ABL_A[(a, d)], "NYU final model (option A)") for a, d, _, _ in ABLATIONS_A]
for abl_id, dataset, R_base, R_new, base_name in runs:
    j = judge(abl_id, dataset, R_base, R_new)
    if abl_id == "ST-D":
        j.update(base_mIoU=None, new_mIoU=None, z_seg=None)
    if abl_id == "ST-S":
        j.update(base_depth=None, new_depth=None, z_depth=None)
    j["baseline"] = base_name
    lb = lambda R: f"{R['log_bias']:+.4f}" if "log_bias" in R else "—"
    j["log_bias (baseline / ablation)"] = f"{lb(R_base)} / {lb(R_new)}"
    rows.append(j)
final_abl = pd.DataFrame(rows).drop(columns=["better", "harm", "kind"])
final_abl.to_csv(FINAL_DIR / "ablations_on_final.csv", index=False)
display(final_abl.style.format(precision=4, na_rep="—"))
print("depth metric: NYU RMSE, Cityscapes AbsRel (512x1024); z > 0 = the ablation is better than its baseline")


,change,dataset,base_depth,new_depth,base_mIoU,new_mIoU,z_depth,z_seg,baseline,log_bias (baseline / ablation)
0,FPE-NYU,nyu,0.4271,0.4199,57.3942,57.9082,2.4048,2.5697,notebook 09 NYU config (frame PE on),+0.0117 / +0.0173
1,ST-D,cityscapes,0.1059,0.1002,—,—,5.3627,—,final model,+0.0574 / +0.0568
2,ST-S,cityscapes,—,—,77.9468,77.7663,—,-0.9026,final model,+0.0574 / +0.8957
3,ST-D,nyu,0.4271,0.3902,—,—,12.3173,—,"notebook 09 NYU config (frame PE on), superseded by §1b",+0.0117 / +0.0078
4,ST-S,nyu,—,—,57.3942,57.0662,—,-1.6402,"notebook 09 NYU config (frame PE on), superseded by §1b",+0.0117 / +0.6631
5,PM,nyu,0.4271,0.4106,57.3942,56.6526,5.5026,-3.7079,"notebook 09 NYU config (frame PE on), superseded by §1b",+0.0117 / +0.0120
6,D4,nyu,0.4271,0.4269,57.3942,56.5983,0.0714,-3.9799,notebook 09 NYU config (frame PE on),+0.0117 / +0.0144
7,ST-D,nyu,0.4205,0.3834,—,—,12.3713,—,NYU final model (option A),+0.0208 / +0.0145
8,ST-S,nyu,—,—,58.2490,58.1088,—,-0.7012,NYU final model (option A),+0.0208 / +0.6637
9,PM,nyu,0.4205,0.4021,58.2490,58.5685,6.1250,1.5975,NYU final model (option A),+0.0208 / +0.0227


depth metric: NYU RMSE, Cityscapes AbsRel (512x1024); z > 0 = the ablation is better than its baseline


## Every ablation of the project
One row per ablation: baseline and changed run, main metrics, z-scores against the NYU seed σ (Cityscapes: the transfer assumption of notebook 09), and the decision.
- **Depth metric:** NYU RMSE; Cityscapes AbsRel at 512×1024.
- **Segmentation metric:** mIoU, single pass.
- The inference-resolution study (notebook 08) has no training run and is shown in that notebook.

In [6]:
def ev(root, name):
    return load_eval(ROOT / root, name)

decisions = {d["decision"]: d for d in load_json(FINAL_DIR / "decisions.json", [])}
ENTRIES = [
    # (id, factor, dataset, baseline (root, name), changed (root, name), decision text)
    ("A/B", "frozen backbone → last 4 blocks fine-tuned", "cityscapes", ("results", "cityscapes_frozen"), ("results", "cityscapes_partial_ft"), "fine-tune 4 blocks"),
    ("A/B", "frozen backbone → last 4 blocks fine-tuned", "nyu", ("results", "nyu_frozen"), ("results", "nyu_partial_ft"), "fine-tune 4 blocks"),
    ("E1", "4 → 8 trainable blocks (critic on, 1 seed)", "nyu", ("results", "nyu_partial_ft"), ("results/exp", "nyu_partial_ft8"), "see E1-seeds"),
    ("E2", "critic on → off", "nyu", ("results", "nyu_partial_ft"), ("results/nocritic", "nyu_partial_ft"), f"critic {decisions.get('critic', {}).get('adopted', '?')}"),
    ("AB1", "standard → frame-consistent position embeddings", "cityscapes", ("results", "cityscapes_partial_ft"), ("results/fpe", "cityscapes_partial_ft"), "adopted on Cityscapes; off on NYU (user, option A, 2026-11-09)"),
    ("AUG-A", "rescale [1, 1.5] + crop 476×630 → no rescale + crop 448×574", "nyu", ("results/seeds/nc8_s42", "nyu_partial_ft8"), ("results/aug/aug_a", "nyu_partial_ft8"), str(decisions.get("rescale / crop (AUG-A vs AUG-B)", {}).get("adopted", "?"))),
    ("AUG-A (4 blocks)", "same as AUG-A, on 4 blocks (stopped first attempt, interaction check)", "nyu", ("results/nocritic", "nyu_partial_ft"), ("results/aug_4blk/aug_a", "nyu_partial_ft"), "not used for the decision"),
    ("AUG-B", "rescale [1, 1.5] → [0.5, 2.0]", "nyu", ("results/seeds/nc8_s42", "nyu_partial_ft8"), ("results/aug/aug_b", "nyu_partial_ft8"), str(decisions.get("rescale / crop (AUG-A vs AUG-B)", {}).get("adopted", "?"))),
    ("AUG-C", "+ rotation ±2.5°", "nyu", ("results/seeds/nc8_s42", "nyu_partial_ft8"), ("results/aug/aug_c", "nyu_partial_ft8"), f"rotate_deg = {decisions.get('rotation (AUG-C)', {}).get('adopted', '?')}"),
    ("AUG-D", "photometric RNG shared by workers → seeded per worker", "nyu", ("results/seeds/nc8_s42", "nyu_partial_ft8"), ("results/aug/aug_d", "nyu_partial_ft8"), "adopted by the rule (neutral); on for Cityscapes, off in the NYU final model (option A, 2026-11-09)"),
]
rows = []
for abl_id, factor, dataset, (rb, nb_), (rn, nn_), decision in ENTRIES:
    Rb, Rn = ev(rb, nb_), ev(rn, nn_)
    if Rb and Rn:
        j = judge(abl_id, dataset, Rb, Rn)
        rows.append(dict(id=abl_id, factor=factor, dataset=dataset, depth_base=j["base_depth"], depth_new=j["new_depth"], z_depth=j["z_depth"],
                         mIoU_base=j["base_mIoU"], mIoU_new=j["new_mIoU"], z_seg=j["z_seg"], decision=decision))

# E1-seeds (means of 3 + 3 seeds, notebook 06)
seed_csv = ROOT / "results" / "compare" / "seeds_nyu_per_run.csv"
if seed_csv.exists():
    df = pd.read_csv(seed_csv)
    a, b = df[df.arm == "4 blocks"], df[df.arm == "8 blocks"]
    fake = lambda d: {"nyu_depth": {"rmse": d["RMSE"].mean()}, "nyu_seg": {"mIoU": d["mIoU"].mean()}}
    j = judge("E1-seeds", "nyu", fake(a), fake(b))
    rows.append(dict(id="E1-seeds", factor="4 → 8 trainable blocks, critic off (mean of 3 seeds each)", dataset="nyu", depth_base=j["base_depth"], depth_new=j["new_depth"],
                     z_depth=j["z_depth"], mIoU_base=j["base_mIoU"], mIoU_new=j["new_mIoU"], z_seg=j["z_seg"],
                     decision=f"blocks = {decisions.get('trainable blocks', {}).get('adopted', '?')}"))

# LOG (notebook 09) and the ablations on the final model (this notebook)
for dataset in ("nyu", "cityscapes"):
    Rb = load_eval(ROOT / "results" / "final" / f"{dataset}_base_s42", f"{dataset}_final")
    Rl = load_eval(ROOT / "results" / "final" / f"{dataset}_log_s42", f"{dataset}_final")
    if Rb and Rl:
        j = judge("LOG", dataset, Rb, Rl)
        rows.append(dict(id="LOG", factor="linear → log-space depth output (vs notebook 09 config, frame PE on)", dataset=dataset, depth_base=j["base_depth"], depth_new=j["new_depth"],
                         z_depth=j["z_depth"], mIoU_base=j["base_mIoU"], mIoU_new=j["new_mIoU"], z_seg=j["z_seg"],
                         decision=f"depth output = {CONFIG['depth_param']}"))
for r in final_abl.to_dict("records"):
    factor = {"ST-D": "multi-task → depth only", "ST-S": "multi-task → segmentation only", "PM": "patch mixing on → off",
              "FPE-NYU": "frame-consistent PE on → off",
              "D4": "sigmoid output → DINOv2 256-bin classification head (side ablation)"}[r["change"]]
    if r["change"] == "FPE-NYU":
        decision = "frame PE off on NYU (user, option A, 2026-11-09)"
    elif "superseded" in r["baseline"]:
        decision = "superseded (baseline had frame PE); see the row vs the NYU final model"
    elif r["change"] == "PM" and "option A" in r["baseline"]:
        decision = ("not adopted (user, 2026-11-11): confirmed on 3 seeds (notebook 12: RMSE 0.4197 -> 0.4036, z +5.4; mIoU null) "
                    "but SwinMTL's per-image mIoU drops 58.43 -> 57.62 (below 58.14); kept on, reported as a trade-off")
    elif r["change"] in ("ST-D", "ST-S"):
        decision = "not adopted: the model stays multi-task (row = cost of multi-task learning)"
    else:
        decision = "final model keeps the setting"
    rows.append(dict(id=r["change"], factor=f"{factor} (vs {r['baseline']})", dataset=r["dataset"], depth_base=r["base_depth"], depth_new=r["new_depth"],
                     z_depth=r["z_depth"], mIoU_base=r["base_mIoU"], mIoU_new=r["new_mIoU"], z_seg=r["z_seg"], decision=decision))

all_abl = pd.DataFrame(rows)
all_abl.to_csv(ROOT / "results" / "compare" / "all_ablations.csv", index=False)
display(all_abl.style.format(precision=4, na_rep="—"))
print("z > 0 = the changed run is better than its baseline; |z| >= 2.5 counts as an effect (decision rule of notebook 09)")


,id,factor,dataset,depth_base,depth_new,z_depth,mIoU_base,mIoU_new,z_seg,decision
0,A/B,frozen backbone → last 4 blocks fine-tuned,cityscapes,0.1259,0.1221,3.0143,74.9901,77.2636,11.3671,fine-tune 4 blocks
1,A/B,frozen backbone → last 4 blocks fine-tuned,nyu,0.4671,0.4329,11.4020,53.8816,57.2734,16.9588,fine-tune 4 blocks
2,E1,"4 → 8 trainable blocks (critic on, 1 seed)",nyu,0.4329,0.4242,2.8812,57.2734,57.7933,2.5994,see E1-seeds
3,E2,critic on → off,nyu,0.4329,0.4282,1.5524,57.2734,57.3912,0.5889,critic off
4,AB1,standard → frame-consistent position embeddings,cityscapes,0.1221,0.1077,11.7867,77.2636,76.9943,-1.3464,"adopted on Cityscapes; off on NYU (user, option A, 2026-11-09)"
5,AUG-A,"rescale [1, 1.5] + crop 476×630 → no rescale + crop 448×574",nyu,0.4205,0.4365,-5.3418,58.2490,56.4518,-8.9861,"unchanged [1, 1.5], crop 476x630"
6,AUG-A (4 blocks),"same as AUG-A, on 4 blocks (stopped first attempt, interaction check)",nyu,0.4282,0.4463,-6.0360,57.3912,55.8699,-7.6064,not used for the decision
7,AUG-B,"rescale [1, 1.5] → [0.5, 2.0]",nyu,0.4205,0.4244,-1.2934,58.2490,58.6394,1.9516,"unchanged [1, 1.5], crop 476x630"
8,AUG-C,+ rotation ±2.5°,nyu,0.4205,0.4183,0.7300,58.2490,58.1729,-0.3808,rotate_deg = 0.0
9,AUG-D,photometric RNG shared by workers → seeded per worker,nyu,0.4205,0.4205,0.0114,58.2490,58.1319,-0.5857,"adopted by the rule (neutral); on for Cityscapes, off in the NYU final model (option A, 2026-11-09)"


z > 0 = the changed run is better than its baseline; |z| >= 2.5 counts as an effect (decision rule of notebook 09)
